## Massachusetts House Roll Call Vote Parser (2009-2017)
This script extracts voting records from MA House roll call PDFs across multiple sessions.

Data Sources:
- Session 186 (2009-2010): Individual PDF files per roll call
- Sessions 187-190 (2011-2017): Annual consolidated PDFs

Processing Methods:
- 2009-2010: Direct text extraction (pdfplumber)
- 2011-2017: OCR-based extraction (pytesseract + pdf2image)

#### 1. Test one page

In [136]:
#%pip install pytesseract
#%pip install pdf2image

In [137]:
import pdfplumber

pdf_path = "HJRollCalls/session_186/session186_rollcall_0238.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    text = page.extract_text()

print(text)


Yea and Nay Supplement
JOURNAL OF THE HOUSE.
Tuesday, October 20, 2009.
Yea and Nay No. 238
On passing to be enacted the engrossed [land taking] Bill authoriz-
ing the lease of certain conservation land in the town of Easton for
agricultural purposes and to establish the Tufts Farm Maintenance
Fund (see House, No. 1885).
YEAS: 154 NAYS: 0
N-V: 6
P=PRESENT X=NOT VOTING *=AFTER VOTE


In [138]:
from pdf2image import convert_from_path
import pytesseract

pdf_path = "HJRollCalls/session_186/session186_rollcall_0456.pdf"

pages = convert_from_path(pdf_path, dpi=300)
img = pages[0]

w, h = img.size
vote_region = img.crop((0, int(h * 0.35), w, int(h * 0.86)))

text = pytesseract.image_to_string(vote_region, config="--psm 6")
print(text)


Y Mr. Speaker Y Donelan Y Linsky Y Speliotis
Y Vallee Y Driscoll Y L' Italien X Spellane
Y Mariano Y Dwyer Y Madden Y Speranzo
Y Haddad Y Dykema Y Malia Y Spiliotis
Y Petrolati Y Ehrlich Y McCarthy N Stanley, H.L.
Y Aguiar Y Fagan Y McMurtry Y Stanley, T.M.
Y Alicea Y Fallon N Miceli Y Story
Y¥ Allen 4 Falzone M4 Michlewitz Y Straus
Y¥ Arciero 4 Fennell 4 Moran Y Sullivan
Y Ashe Y Fernandes Y Murphy, C.A. N Swan
Y Atkins Y Ferrante XY Murphy, J. Y Timilty
Y Atsalis Y Finegold Y Murphy, K.J. Y Tobin
Y Ayers Y Flynn Y Nangle Y Toomey
Y Balser xX Forry Y Naughton Y Torrisi
Y Basile Y Fox Y O'Day Y Turner
Y Benson Y Fresolo Y O'Flaherty Y Wagner
Y¥ Binienda 4 Galvin 4 Patrick Y Wallace
X Bosley Y Garballey Y Peake Y Walsh, M.J.

Y Bowles Y Garry Y Pedone Y Walsh, S.
Y Bradley Y Gobi Y Peisch Y Walz
Y Brady Y Golden N Pignatelli N Welch
Y Brownsberger Y Grant Y Provost Y Wolf
Y Cabral Y Greene Y Puppolo Y Jones
Y Callahan ¥ Gregoire XY Quinn Y Peterson

N Calter Y Guyer Y Reinstein N Hill
Y

#### 2. House Roll Call in the 186th Session 

In [10]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import re
import pandas as pd
from PIL import Image


In [ ]:
def extract_header_info(page):
    """
    Extract metadata from the top portion of a roll call page.
    """
    text = page.extract_text() or ""
    #print(text)  
    lines = text.splitlines()

    # Date
    date_match = re.search(r"[A-Za-z]+,\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}", text)
    date = date_match.group(0) if date_match else None

    # Rollcall number
    rc_match = re.search(r"Yea and Nay No\.\s*(\d+)", text)
    rollcall_number = int(rc_match.group(1)) if rc_match else None

    # Bill number\
    bill_match = re.search(r"\((?:see\s+)?(House|Senate),\s+No\.\s+(\d+)(?:[^)]*)\)", text, re.I)
    if bill_match:
        prefix = "H" if bill_match.group(1).lower() == "house" else "S"
        bill_number = f"{prefix}.{bill_match.group(2)}"
    else:
        bill_number = None

    # Amendment
    amendment_flag = "amended" if re.search(r"amended", text, re.I) else None
  
    # TitleText
    if len(lines) > 7:
        title_lines = lines[4:-3]
        title_text = " ".join(l.strip() for l in title_lines if l.strip())
    else:
        title_text = None

    # Count：YEAS NAYS N-V
    yeas_match = re.search(r"YEAS:\s*(\d+)", text)
    nays_match = re.search(r"NAYS:\s*(\d+)", text)
    nv_match   = re.search(r"N[-–]V:\s*(\d+)", text)

    yeas = int(yeas_match.group(1)) if yeas_match else None
    nays = int(nays_match.group(1)) if nays_match else None
    nv   = int(nv_match.group(1))   if nv_match   else None

    return {
        "Date": date,
        "RollCallNumber": rollcall_number,
        "BillNumber": bill_number,
        "AmendmentNumber": amendment_flag,
        "TitleText": title_text,
        "YeasCount": yeas,
        "NaysCount": nays,
        "NVCount": nv,
    }


In [ ]:
def ocr_vote_region(pdf_path, page_index=0):
    """
    Convert PDF page to image and OCR the voting section.
   
    """
    images = convert_from_path(pdf_path, dpi=300)
    img = images[page_index]

    w, h = img.size
    vote_region = img.crop((0, int(h * 0.35), w, int(h * 0.86)))

    # OCR
    text = pytesseract.image_to_string(vote_region, config="--psm 6")

    # Debug
    #print("\n===== OCR RAW TEXT (Debug) =====")
    #print(text)
    #print("================================\n")

    return text


In [ ]:
VOTE_TOKENS = {"Y", "N", "X", "P", "XY", "xX", "¥", "XxX"}

def is_vote_token(tok):
    tok = tok.strip()
    if not tok:
        return False
    
    if tok in VOTE_TOKENS:
        return True
    
    if "¥" in tok:
        return True
    
    if len(tok) == 1:
        if tok.isdigit() or tok.upper() in {"Y", "N", "X", "P"}:
            return True
        return False

    # Definite non-votes
    if tok.endswith("."):     # J. / C.A. / T.M.
        return False
    if "," in tok:            # Murphy, / O'Day,
        return False
    if "'" in tok or "'" in tok:  # O'Day / L' Italien
        return False


    # Two-character combinations (XY, XX, etc.)
    if len(tok) == 2:
        if tok.isalpha():
            if tok.upper() in ("XY", "XX", "YX", "YY", "XP", "PX"):
                return True
            return False
        
        return False

    return False


In [ ]:
def split_line_to_votes(line: str):
    #raw_tokens = line.split()
    tokens = line.split()
    results = []
    i = 0

    while i < len(tokens):
        tok = tokens[i]

        if is_vote_token(tok):
            vote = tok.upper()

            j = i + 1
            name_parts = []
            while j < len(tokens) and not is_vote_token(tokens[j]):
                name_parts.append(tokens[j])
                j += 1

            name = " ".join(name_parts).strip()
            if name:            
                results.append((name, vote))

            i = j              
        else:
            i += 1

    return results



In [38]:
def parse_votes(ocr_text: str):
    rows = []
    for raw_line in ocr_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        pairs = split_line_to_votes(line)
        rows.extend(pairs)

    return rows


In [ ]:
def process_single_rollcall(pdf_path, general_court_number: int):
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        header = extract_header_info(page)
        header["GeneralCourtNumber"] = general_court_number

    ocr_text = ocr_vote_region(pdf_path)
    vote_pairs = parse_votes(ocr_text)

    records = []
    for name, vote in vote_pairs:
        rec = {
            **header,
            "RollCallName": name,
            "Vote": vote,
        }
        records.append(rec)

    return pd.DataFrame(records)


In [83]:
"""-----single pdf test------

df_single = process_single_rollcall(
    "HJRollCalls/session_186/session186_rollcall_0034.pdf",
    GENERAL_COURT_NUMBER
)
print(len(df_single))
df_single.head()
"""


'-----single pdf test------\n\ndf_single = process_single_rollcall(\n    "HJRollCalls/session_186/session186_rollcall_0034.pdf",\n    GENERAL_COURT_NUMBER\n)\nprint(len(df_single))\ndf_single.head()\n'

In [ ]:
import os
import pandas as pd

def process_entire_session(folder_path, general_court_number):
    all_records = []

    files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".pdf")
    ])

    print(f"find {len(files)} rollcall PDF files")

    for f in files:
        pdf_path = os.path.join(folder_path, f)
        print(f"\n---- Processing {f} ----")

        try:
            df_single = process_single_rollcall(pdf_path, general_court_number)
            all_records.append(df_single)
        except Exception as e:
            print(f"Error processing {f}: {e}")

    if all_records:
        df_all = pd.concat(all_records, ignore_index=True)
        return df_all
    else:
        print("⚠️ No rollcall records found.")
        return pd.DataFrame()


In [130]:
folder = "HJRollCalls/session_186"
GENERAL_COURT_NUMBER = 186

df_186 = process_entire_session(folder, GENERAL_COURT_NUMBER)

print(df_186.head())
print(len(df_186))


find 513 rollcall PDF files

---- Processing session186_rollcall_0001.pdf ----

---- Processing session186_rollcall_0002.pdf ----

---- Processing session186_rollcall_0003.pdf ----

---- Processing session186_rollcall_0004.pdf ----

---- Processing session186_rollcall_0005.pdf ----

---- Processing session186_rollcall_0006.pdf ----

---- Processing session186_rollcall_0007.pdf ----

---- Processing session186_rollcall_0008.pdf ----

---- Processing session186_rollcall_0009.pdf ----

---- Processing session186_rollcall_0010.pdf ----

---- Processing session186_rollcall_0011.pdf ----

---- Processing session186_rollcall_0012.pdf ----

---- Processing session186_rollcall_0013.pdf ----

---- Processing session186_rollcall_0014.pdf ----

---- Processing session186_rollcall_0015.pdf ----

---- Processing session186_rollcall_0016.pdf ----

---- Processing session186_rollcall_0017.pdf ----

---- Processing session186_rollcall_0018.pdf ----

---- Processing session186_rollcall_0019.pdf ----

--

In [ ]:
#df_186["Vote"].unique()

array(['N', 'X', 'Y', 'XX', 'XXX', '4', '¥', 'XY', 'Y¥', 'YX', '¥Y', 'P',
       '¥*ST.'], dtype=object)

In [133]:
#df_186[df_186["Vote"] == "4"]


#### 3. House Roll Call from 2011 to 2017

In [ ]:
def ocr_vote_region_from_page(page):
    img = page.to_image(resolution=300).original
    w, h = img.size
    vote_region = img.crop((0, int(h * 0.35), w, int(h * 0.86)))
    text = pytesseract.image_to_string(vote_region, config="--psm 6")
    return text

In [ ]:
def process_single_rollcall_page(page, general_court_number:int):
    
    header = extract_header_info(page)
    header["GeneralCourtNumber"] = general_court_number
    ocr_text = ocr_vote_region_from_page(page)
    vote_pairs = parse_votes(ocr_text)

    records = []
    for name, vote in vote_pairs:
        records.append({
            **header,
            "RollCallName": name,
            "Vote": vote,
        })

    return pd.DataFrame(records)

In [ ]:
def is_blank_page(page):
    text = page.extract_text()
    if not text:
        return True
    if len(text.strip()) < 10: 
        return True
    return False


def process_annual_rollcalls(pdf_path, general_court_number):
    all_dfs = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"📄 Total pages detected: {total_pages}")

        # Skip cover page
        start_page = 1  

        # Auto-skip blank second page if present
        if start_page < total_pages and is_blank_page(pdf.pages[start_page]):
            print("Blank second page detected → skipping")
            start_page += 1

        # Process each roll call page
        for page_num in range(start_page, total_pages):
            page = pdf.pages[page_num]
            print(f"Processing page {page_num+1}/{total_pages}")

            try:
                df_page = process_single_rollcall_page(page, general_court_number)
                if isinstance(df_page, pd.DataFrame) and not df_page.empty:
                    all_dfs.append(df_page)
                else:
                    print(f"⚠ Warning: Page {page_num+1} returned empty or invalid data")
            except Exception as e:
                print(f"❗ Error processing page {page_num+1}: {e}")
                import traceback
                traceback.print_exc()

    if all_dfs:
        df = pd.concat(all_dfs, ignore_index=True)
        return df
    else:
        return pd.DataFrame()



In [120]:
df_2011 = process_annual_rollcalls(
    "HJRollCalls/2011/187_2011_RollCalls.pdf",
    general_court_number=187
)

print(df_2011.head())
print("Total vote records:", len(df_2011))


📄 Total pages detected: 183
Blank second page detected → skipping
Processing page 3/183
Processing page 4/183
Processing page 5/183
Processing page 6/183
Processing page 7/183
Processing page 8/183
Processing page 9/183
Processing page 10/183
Processing page 11/183
Processing page 12/183
Processing page 13/183
Processing page 14/183
Processing page 15/183
Processing page 16/183
Processing page 17/183
Processing page 18/183
Processing page 19/183
Processing page 20/183
Processing page 21/183
Processing page 22/183
Processing page 23/183
Processing page 24/183
Processing page 25/183
Processing page 26/183
Processing page 27/183
Processing page 28/183
Processing page 29/183
Processing page 30/183
Processing page 31/183
Processing page 32/183
Processing page 33/183
Processing page 34/183
Processing page 35/183
Processing page 36/183
Processing page 37/183
Processing page 38/183
Processing page 39/183
Processing page 40/183
Processing page 41/183
Processing page 42/183
Processing page 43/18

In [ ]:
import os

def process_years(root_folder="HJRollCalls", start_year=2011, end_year=2017):
    all_year_results = []

    for year in range(start_year, end_year + 1):
        print(f"\n===== Processing year {year} =====")

        # HJRollCalls/2011/187_2011_RollCalls.pdf
        folder = os.path.join(root_folder, str(year))
        files = os.listdir(folder)
        
        pdf_file = [f for f in files if f.endswith("RollCalls.pdf")]
        if not pdf_file:
            print(f"⚠ No rollcall file found for year {year}")
            continue
        
        pdf_path = os.path.join(folder, pdf_file[0])
        
        gc_number = int(pdf_file[0].split("_")[0])

        print(f"Found file: {pdf_file[0]} (General Court = {gc_number})")

        df_year = process_annual_rollcalls(pdf_path, gc_number)
        
        df_year["Year"] = year
        
        all_year_results.append(df_year)

    final_df = pd.concat(all_year_results, ignore_index=True)
    return final_df


In [123]:
df_all = process_years("HJRollCalls", 2011, 2017)



===== Processing year 2011 =====
Found file: 187_2011_RollCalls.pdf (General Court = 187)
📄 Total pages detected: 183
Blank second page detected → skipping
Processing page 3/183
Processing page 4/183
Processing page 5/183
Processing page 6/183
Processing page 7/183
Processing page 8/183
Processing page 9/183
Processing page 10/183
Processing page 11/183
Processing page 12/183
Processing page 13/183
Processing page 14/183
Processing page 15/183
Processing page 16/183
Processing page 17/183
Processing page 18/183
Processing page 19/183
Processing page 20/183
Processing page 21/183
Processing page 22/183
Processing page 23/183
Processing page 24/183
Processing page 25/183
Processing page 26/183
Processing page 27/183
Processing page 28/183
Processing page 29/183
Processing page 30/183
Processing page 31/183
Processing page 32/183
Processing page 33/183
Processing page 34/183
Processing page 35/183
Processing page 36/183
Processing page 37/183
Processing page 38/183
Processing page 39/183

In [ ]:
#print(df_all.shape)
#print(df_all["Vote"].unique())

(270511, 12)
['Y' 'X' 'XY' 'XXX' 'XX' 'YX' '¥' 'Y¥' 'N' 'P' '4' '0' 'YY' '¥:' '¥.' 'X¥']


In [129]:
#df_all[df_all["Vote"] == "XX"]

In [134]:
df_combined = pd.concat([df_186, df_all], ignore_index=True)

print(f"Total records after merging: {len(df_combined)}")
print(f"df_186 records: {len(df_186)}")
print(f"df_all records: {len(df_all)}")
print("\nCombined data preview:")
print(df_combined.head())
print(df_combined.tail())

df_combined.to_csv("house_rollcalls_2009_2017.csv", index=False)
print("\n✅ Saved to 'house_rollcalls_2009_2017.csv'")


Total records after merging: 351758
df_186 records: 81247
df_all records: 270511

Combined data preview:
                          Date  RollCallNumber BillNumber AmendmentNumber  \
0  Wednesday, January 14, 2009               1       H.98            None   
1  Wednesday, January 14, 2009               1       H.98            None   
2  Wednesday, January 14, 2009               1       H.98            None   
3  Wednesday, January 14, 2009               1       H.98            None   
4  Wednesday, January 14, 2009               1       H.98            None   

                                           TitleText YeasCount NaysCount  \
0  On adoption of an amendment (offered by Mr. Jo...        20       135   
1  On adoption of an amendment (offered by Mr. Jo...        20       135   
2  On adoption of an amendment (offered by Mr. Jo...        20       135   
3  On adoption of an amendment (offered by Mr. Jo...        20       135   
4  On adoption of an amendment (offered by Mr. Jo...

In [135]:
df_combined.columns

Index(['Date', 'RollCallNumber', 'BillNumber', 'AmendmentNumber', 'TitleText',
       'YeasCount', 'NaysCount', 'NVCount', 'GeneralCourtNumber',
       'RollCallName', 'Vote', 'Year'],
      dtype='object')